In [1]:
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():  # skip empty lines
                data.append(json.loads(line))
    return data

classification_data=load_jsonl("/content/synthetic_data_for_classification (3).jsonl")
contrastive_data=load_jsonl("/content/synthetic_data_for_contrastive_learning (1).jsonl")

In [14]:
#converting both data sets to binary form
#converting classification data to binary form
def convert_classification(data):
    rows = []

    for item in data:
        anchor = item["anchor_text"]
        A = item["text_a"]
        B = item["text_b"]
        a_closer = item["text_a_is_closer"]

        if anchor is None or A is None or B is None:
            continue

        if a_closer:
            rows.append({
                "text": anchor + " [SEP] " + A,
                "label": 1
            })
            rows.append({
                "text": anchor + " [SEP] " + B,
                "label": 0
            })
        else:
            rows.append({
                "text": anchor + " [SEP] " + B,
                "label": 1
            })
            rows.append({
                "text": anchor + " [SEP] " + A,
                "label": 0
            })

    return rows


In [16]:
#converting contrastive data
def convert_contrastive(data):
  rows=[]

  for item in data:
    anchor=item["anchor_story"]
    similar = item["similar_story"]
    dissimilar = item["dissimilar_story"]

    if not anchor or not similar or not dissimilar:
            continue

    rows.append({
        "text":anchor+"[SEP]"+ similar,
        "label":1
    })
    rows.append({
        "text":anchor+"[SEP]"+dissimilar,
        "label":0
    })

  return rows;

In [17]:
#combining both
contrastive_rows = convert_contrastive(contrastive_data)
classification_rows = convert_classification(classification_data)

all_rows = contrastive_rows + classification_rows
df = pd.DataFrame(all_rows)


In [18]:
df.head()

,text,label
0,"A mysterious individual, known only by their a...",1
1,"A mysterious individual, known only by their a...",0
2,A mysterious drifter arrives in the lawless fr...,1
3,A mysterious drifter arrives in the lawless fr...,0
4,"A team of paranormal investigators, led by sea...",1


In [19]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words="english"
)

X = tfidf.fit_transform(df["text"])
y = df["label"]


In [20]:
#logistic regression
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X, y)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [21]:
#here we are verifying if the model is behaving properlly
preds = model.predict(X)
print(classification_report(y, preds))



              precision    recall  f1-score   support

           0       0.97      0.97      0.97      3794
           1       0.97      0.97      0.97      3794

    accuracy                           0.97      7588
   macro avg       0.97      0.97      0.97      7588
weighted avg       0.97      0.97      0.97      7588



from the result we understand that
class 1 (similar):
Precision = 0.97
Means that when the model says “these stories are similar”, it is correct 97% of the time

For class 0 (dissimilar):
Precision = 0.97
Means that when the model says “these stories are dissimilar”, it is correct 97% of the time

similarly for recall

In [30]:
def predict_trackA(anchor,A,B,threshold=0.5):
    text_A = anchor + " [SEP] " + A
    text_B = anchor + " [SEP] " + B

    X_test=tfidf.transform([text_A,text_B])
    probs=model.predict_proba(X_test)[:,1]

    prob_A,prob_B=probs

    if prob_A<threshold and prob_B<threshold:
      return "dissimilar"
    elif prob_A>prob_B:
      return "A"
    else:
      return "B"

In [33]:
#testing
anchor_text = """A mysterious individual, known only by their alias "The Wanderer", arrives in the small, rural town of Willow Creek, sparking curiosity and intrigue among its residents. ..."""

text_a = """In the coastal city of Tidal Cove, a reclusive figure known as "The Architect" emerges, captivating the attention of its residents. ..."""

text_b = """In the secluded hamlet of Ravenshire, a mysterious individual known as "The Traveler" emerges, captivating the attention of its residents. ..."""
result = predict_trackA(
    anchor=anchor_text,
    A=text_a,
    B=text_b
)

print(result)


B


In [36]:
##testing on dev data


from sklearn.metrics import classification_report

def convert_dev_for_eval(dev_data):
    texts = []
    labels = []

    for item in dev_data:
        anchor = item.get("anchor_text")
        A = item.get("text_a")
        B = item.get("text_b")
        a_closer = item.get("text_a_is_closer")

        if anchor is None or A is None or B is None or a_closer is None:
            continue

        if a_closer:
            texts.append(anchor + " [SEP] " + A)
            labels.append(1)
            texts.append(anchor + " [SEP] " + B)
            labels.append(0)

        else:
            texts.append(anchor + " [SEP] " + B)
            labels.append(1)
            texts.append(anchor + " [SEP] " + A)
            labels.append(0)

    return texts, labels


X_dev_texts, y_dev = convert_dev_for_eval(dev_data)
print("Number of dev samples:", len(X_dev_texts))

X_dev = tfidf.transform(X_dev_texts)

dev_preds = model.predict(X_dev)

print("\nDevelopment Set Performance:")
print(classification_report(y_dev, dev_preds))


Number of dev samples: 400

Development Set Performance:
              precision    recall  f1-score   support

           0       0.25      0.01      0.01       200
           1       0.50      0.98      0.66       200

    accuracy                           0.49       400
   macro avg       0.37      0.49      0.34       400
weighted avg       0.37      0.49      0.34       400

